# Gebäudesektormodell: Jahresergebnisse und Browser-Quick-Check

Dieser Ablauf verwendet ausschließlich die 217 annual-Dateien unter
`data/exports/typologies/building_sector`, ihren Checkpoint und die SCEN-Definitionen.
Es werden keine Simulationen gestartet. Die ursprünglichen Dateien bleiben erhalten.

Priorität dieses POC ist die vollständige technische Kette bis zur statischen Browseransicht.
Fachliche Auffälligkeiten werden einmal dokumentiert, ohne Korrektur- oder Nachsimulationsschleifen.
Die Diagramme zeigen ungewichtete Varianten; daraus folgt keine Verteilung des österreichischen Gebäudebestands.

Voraussetzungen: `pip install -e ".[publication,dev]"` im Repository.


In [1]:
from pathlib import Path
import sys, json
from IPython.display import display, IFrame
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/pexl").is_dir())
sys.path.insert(0, str(ROOT / "scripts"))
from pexl import Project, reporting
from pexl.plot import plotly
from build_building_sector_results import build, OUTPUT, diagnostics, digest
from assessment_explained import build as build_quickcheck
WORK = ROOT / "notebooks/local_exports/building_sector"
WORK.mkdir(parents=True, exist_ok=True)


## Konsolidierung und technische Abnahme

`REBUILD=True` führt den vollständigen Ablauf aus: Quelldateien und Checkpoint-Hashes,
eindeutige Szenarionamen, IN-/OUT-Spalten, SCEN-Parameter und Schema-Variablen prüfen;
anschließend `Project.from_files(unknown="raise")`, `to_excel(include_out=True)` und
`from_excel(unknown="raise")`. Alle vorhandenen Werte werden im Roundtrip verglichen
(rtol 1e-12, atol 1e-10, ausschließlich gegen Excel-Serialisierungsrundung).
Bei der normalen erneuten Auswertung wird der bereits geprüfte Export geladen.
Der Prüfbericht ist ein reproduzierbares Protokoll im ignorierten Ausgabeverzeichnis,
keine zweite fachliche Datenquelle.


In [2]:
REBUILD = False
audit_path = WORK / "validation.json"
if REBUILD or not OUTPUT.exists() or not audit_path.exists():
    project, audit = build(audit_output=audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    assert digest(OUTPUT) == audit["output_sha256"], "Export stimmt nicht mit Prüfprotokoll überein"
    project = Project.from_excel(OUTPUT, unknown="raise")
assert len(project) == 5400
display({k: audit[k] for k in ("files", "scenarios", "schema", "base_reload", "roundtrip_checked_values", "seconds")})


{'files': 217,
 'scenarios': 5400,
 'schema': '1_13_1_dev',
 'base_reload': 'first_only',
 'roundtrip_checked_values': 6458400,
 'seconds': 484.8434728999855}

## Einmalige Plausibilitätsnotiz

Nicht-null Residuen sind Hinweise, keine automatische Fehlerklassifikation.
Die Schwelle 1e-6 dient hier nur der numerischen Sichtung und ist keine fachliche Abnahmetoleranz.
Die verfügbaren Jahresdaten können einen Zustandsübertrag bei `first_only` weder sicher beweisen
noch ausschließen. Reihenfolge, Hashes und Restgrößen sind dokumentiert; es werden keine weiteren
Simulationen oder Prüfzyklen ausgelöst.

Im vorliegenden Stand verwendet das Diagramm negative Dichtebeiträge betragsmäßig auf der Bedarfsseite,
während die exportierte virtuelle Bedarfssumme sie negativ addiert. Der Originalsaldo wird nicht korrigiert.
Die Kontextansicht schreibt jede dieser vorhandenen Größen mit ihrer jeweiligen Vorzeichenkonvention fort.
COM-Fehlercodes in nicht verwendeten Kennwerten bleiben im Projekt erhalten und stehen im Prüfprotokoll.


In [3]:
display(diagnostics(project))
display({"fehlende_Werte": audit["missing_values"],
         "gespeicherte_COM_Fehlercodes": audit["excel_error_sentinels"]})


,variable,missing,nonzero,max_abs
0,test_heat_balance_winter,0,5400,1.563983e+00
1,test_heat_balance_summer,0,1200,1.510299e+00
2,test_heat_balance,0,531,2.452250e+00
3,test_heat_balance_seasonal_mismatch,0,5400,1.910923e+00
4,test_EUI_balance,0,0,2.941647e-12
5,PEI_virtual_balance_test,0,1620,4.410980e+01
6,PEI_importexport_balance_test,0,0,2.165734e-11


{'fehlende_Werte': {'preset_recorded_renovation': 5400,
  'building_count': 5400,
  'preset_recorded_vent_modes': 5400,
  'preset_recorded_vent_night_modes': 5400,
  'vent_scale_other': 5400,
  'preset_recorded_heating_emission_system': 5400,
  'preset_recorded_cooling_modes': 5400,
  'preset_recorded_PV_system': 5400,
  'preset_recorded_flexibility_thermal': 5400,
  'preset_recorded_mobility': 5400,
  'preset_recorded_construction_type': 5400,
  'GWP_refratio_walls': 5400,
  'ghg_conversion_factor_gasoline': 5400,
  'cost_total': 5400},
 'gespeicherte_COM_Fehlercodes': {'Plugloads_scale_other': 5400,
  'Ta_avg_cooling_period': 4200,
  'Tc_avg_winter': 5400,
  'Tc_avg_summer': 5400,
  'dTc_winter': 5400,
  'dTc_summer': 5400,
  'GWP_emInt_flex': 2700,
  'GWP_emInt_batt_charge': 5400,
  'GWP_emInt_ev_charge': 5400,
  'GWP_emInt_PV_feedin': 1620,
  'PV_own_consumption_direct': 1620,
  'PV_own_consumption_flex': 1620,
  'PV_own_consumption': 1620}}

## Filtergrundlagen und Verteilungen

Die PV-Ausstattung kommt aus `PV_roof_share` (Einheit ratio); `0.3` bedeutet 30 %.
`preset_recorded_PV_system` ist im Bestand teilweise leer und wird nicht als Ersatz für die Fläche benutzt.
Heizungsfamilien werden aus dem aufgezeichneten Heizungslabel gebildet, Detailwerte bleiben erhalten.


In [4]:
fields = ["preset_recorded_geometry", "preset_recorded_hulls",
          "preset_recorded_heating_system", "preset_recorded_flexibility_system",
          "PV_roof_share", "preset_recorded_PV_system", "GFA_total", "FSI",
          "QH", "QC", "PEI_balance", "PEI_virtual_demand", "PEI_virtual_supply"]
frame = project.view.select(*fields).to_frame()
display(frame.head())
display(frame[["PV_roof_share", "preset_recorded_PV_system"]].value_counts(dropna=False))


,preset_recorded_geometry,preset_recorded_hulls,preset_recorded_heating_system,preset_recorded_flexibility_system,PV_roof_share,preset_recorded_PV_system,GFA_total,FSI,QH,QC,PEI_balance,PEI_virtual_demand,PEI_virtual_supply
column_name,,,,,,,,,,,,,
EFH_1G|vor1919|WP_Sole|PV30|noFlex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),Keine Flexibilität,0.3,NaN,133.875,0.666667,481.553227,0,-194.067660,273.871202,123.913338
EFH_1G|vor1919|WP_Sole|PV30|Flex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),PV + Signal,0.3,NaN,133.875,0.666667,520.197998,0,-127.127506,292.810107,209.792398
EFH_1G|vor1919|WP_Sole|PV70|noFlex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),Keine Flexibilität,0.7,NaN,133.875,0.666667,481.553227,0,-28.849876,273.871202,289.131123
EFH_1G|vor1919|WP_Sole|PV70|Flex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),PV + Signal,0.7,NaN,133.875,0.666667,531.833454,0,22.813976,298.454774,365.378546
EFH_1G|vor1919|WP_Sole|PV90|noFlex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),Keine Flexibilität,0.9,NaN,133.875,0.666667,481.553227,0,53.759017,273.871202,371.740015


PV_roof_share  preset_recorded_PV_system
0.3            NaN                          1800
0.7            NaN                          1800
0.9            NaN                          1800
Name: count, dtype: int64

In [5]:
display(plotly.scenario_distribution(project.view, "QH", by="preset_recorded_hulls"))
display(plotly.scenario_distribution(project.view, "QC", by="preset_recorded_geometry", kind="violin"))


## Bedarf und Deckung, Dichte und ausgewählte Bilanzen

Der Scatter verwendet die exportierten virtuellen Summen. Wegen der dokumentierten
Vorzeichenabweichung ist er separat von der betragsmäßigen Reportdarstellung zu interpretieren.
BGF dient hier der Markergröße, nicht einer Bestandsgewichtung.


In [6]:
display(plotly.scenario_scatter(project.view, "PEI_virtual_demand", "PEI_virtual_supply",
    color="preset_recorded_heating_system", symbol="preset_recorded_geometry",
    size="GFA_total", hover_data=["FSI", "PV_roof_share", "preset_recorded_hulls"]))
display(plotly.scenario_scatter(project.view, "FSI", "PEI_balance",
    color="preset_recorded_heating_system", hover_data=["QH", "QC"]))

display(plotly.report_balance_scatter(project.view, "final_energy_balance", "FSI", color="preset_recorded_heating_system"))

In [7]:
# Exakte Namen können hier durch interessante Treffer aus frame ersetzt werden.
selected_names = project.column_names()[:6]
selected = Project()
for name in selected_names:
    selected.add_scenario(project[name])
for chart in ["heat_balance", "final_energy_balance", "primary_energy_balance", "gwp_ee"]:
    display(plotly.chart(selected.view, chart))


## Hinweise zu Variantenähnlichkeit und kontrollierten Vergleichen

Identische QH-/QC-Werte werden lediglich gezählt. Unterschiede zwischen PV- und Flexvarianten
dürfen erst nach Kontrolle der übrigen IN-Parameter kausal interpretiert werden.
Diese Tabelle ermöglicht die Auswahl interessanter Namen für den obigen Vergleich;
sie enthält keine automatisierte Bewertung der Simulationen.


In [8]:
display(frame.groupby(["QH", "QC"], dropna=False).size().sort_values(ascending=False).head(12).rename("Variantenanzahl"))
display(frame.loc[frame["preset_recorded_geometry"] == frame.iloc[0]["preset_recorded_geometry"]].head(12))


QH          QC
423.268201  0     45
415.510278  0     45
21.938328   0     45
12.921230   0     45
18.640220   0     45
160.824179  0     45
158.822324  0     45
383.354356  0     45
412.229619  0     45
342.737776  0     45
27.088175   0     45
26.383401   0     45
Name: Variantenanzahl, dtype: int64

,preset_recorded_geometry,preset_recorded_hulls,preset_recorded_heating_system,preset_recorded_flexibility_system,PV_roof_share,preset_recorded_PV_system,GFA_total,FSI,QH,QC,PEI_balance,PEI_virtual_demand,PEI_virtual_supply
column_name,,,,,,,,,,,,,
EFH_1G|vor1919|WP_Sole|PV30|noFlex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),Keine Flexibilität,0.3,NaN,133.875,0.666667,481.553227,0,-194.067660,273.871202,123.913338
EFH_1G|vor1919|WP_Sole|PV30|Flex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),PV + Signal,0.3,NaN,133.875,0.666667,520.197998,0,-127.127506,292.810107,209.792398
EFH_1G|vor1919|WP_Sole|PV70|noFlex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),Keine Flexibilität,0.7,NaN,133.875,0.666667,481.553227,0,-28.849876,273.871202,289.131123
EFH_1G|vor1919|WP_Sole|PV70|Flex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),PV + Signal,0.7,NaN,133.875,0.666667,531.833454,0,22.813976,298.454774,365.378546
EFH_1G|vor1919|WP_Sole|PV90|noFlex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),Keine Flexibilität,0.9,NaN,133.875,0.666667,481.553227,0,53.759017,273.871202,371.740015
EFH_1G|vor1919|WP_Sole|PV90|Flex,EFH_1G,MFH_bis1919,Wärmepumpe (Erdwärme),PV + Signal,0.9,NaN,133.875,0.666667,534.563207,0,101.738264,299.826553,445.674613
EFH_1G|vor1919|WP_Wasser|PV30|noFlex,EFH_1G,MFH_bis1919,Wärmepumpe (Grundwasser),Keine Flexibilität,0.3,NaN,133.875,0.666667,481.553227,0,-170.174811,249.978354,123.913338
EFH_1G|vor1919|WP_Wasser|PV30|Flex,EFH_1G,MFH_bis1919,Wärmepumpe (Grundwasser),PV + Signal,0.3,NaN,133.875,0.666667,522.132054,0,-109.603848,268.047311,202.553260
EFH_1G|vor1919|WP_Wasser|PV70|noFlex,EFH_1G,MFH_bis1919,Wärmepumpe (Grundwasser),Keine Flexibilität,0.7,NaN,133.875,0.666667,481.553227,0,-4.957027,249.978354,289.131123


## Statischen Quick-Check erzeugen

Der Generator verwendet das bereits geladene Project und die bestehenden Reports und Plotting-Funktionen.
Die Browserdaten enthalten gemeinsame Diagrammvorlagen und Jahreswerte für alle 5.400 Szenarien.
Die fünf Analyseansichten zeigen wahlweise Filtertreffer oder Auswahl. Es gibt keine feste Auswahlgrenze.
Oberhalb von 100 Szenarien zeigen kompakte WebGL-Linien alle einzelnen Bilanzbeiträge; Listen und Tabellen sind paginiert.


In [9]:
page = build_quickcheck(WORK / "quickcheck.html", source=OUTPUT, project=project)
display(IFrame(src="local_exports/building_sector/quickcheck.html", width="100%", height=900))


{"scenarios": 5400, "payload_bytes": 3455648, "payload_gzip_bytes": 363002, "html_bytes": 8330082, "html_gzip_bytes": 1838756}


## Browser-Abnahme und Weiterverwendung

`node tests/quickcheck_browser.cjs website/assets/assessment-explained/index.html notebooks/local_exports/building_sector/browser`
prüft echte Browserausführung, Filter, Auswahlgrenze, Kontextberechnung und Rendering mit 1/5/10/20 Varianten.
Playwright muss installiert sein; optional PLAYWRIGHT_MODULE und CHROME_EXECUTABLE setzen.
Die Messung umfasst Desktop und mobile Viewport-Emulation auf derselben Desktop-CPU.
Sie ersetzt keine Messung auf einem physischen Mobilgerät oder über das spätere Hosting.

Für peq-higrube dieselbe HTML-Datei nach assets/assessment/index.html übernehmen und per iframe
einbinden (siehe docs/development/assessment_explained.md). Keine zweite Implementierung.
Veröffentlichung bleibt ein gesonderter Schritt.


In [10]:
benchmark = WORK / "browser/browser-performance.json"
if benchmark.exists():
    results = json.loads(benchmark.read_text(encoding="utf-8"))
    display([{k: v for k, v in r.items() if k != "contextSample"} for r in results["results"]])
else:
    print("Browsermessung separat mit dem dokumentierten Testskript ausführen.")


[{'profile': 'desktop', 'loadMs': 1855.611, 'scenarios': 5400, 'initializationMs': 642.5999999642372, 'filterMedianMs': 4.099999964237213, 'filterP95Ms': 6.899999976158142, 'renderMs': {'1': [164.5], '5': [168.80000001192093], '10': [256.89999997615814], '20': [323.80000001192093], '21': [240.10000002384186], '100': [898], '5400': [533.6000000238419]}, 'heap': {'used': 81400000, 'total': 148000000}, 'userAgent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/156.0.0.0 Safari/537.36'}, {'profile': 'mobile viewport emulation (desktop CPU)', 'loadMs': 1778.1836000000003, 'scenarios': 5400, 'initializationMs': 542.9000000357628, 'filterMedianMs': 4.399999976158142, 'filterP95Ms': 5.300000011920929, 'renderMs': {'1': [138.69999998807907], '5': [145.5], '10': [183.4000000357628], '20': [189], '21': [161.5999999642372], '100': [529.3000000119209], '5400': [426.7000000476837]}, 'heap': {'used': 81400000, 'total': 148000000}, 'userAgent': 'Mozil